# Databricks Notebook: 21_silver_events

**Purpose:** Clean, standardise and lightly enrich events. Non-intrusive quality gate: valid rows MERGE into `silver.events`, failing rows go to `silver.events_quarantine`.

### Target Tables:
- `silver.events` (one row per unique `event_id`; MERGE; Change Data Feed on)
- `silver.events_quarantine` (append-only; original columns + `_dq_rule`, `_dq_reason`, `_quarantined_at`)

### Key Transformations:
1. Latest delivery per `_partition_dt`, then dedup on `event_id`.
2. Casts (`played_at` UTC timestamp, `ms_played` INT, `skipped` BOOLEAN), trimming, `source` `real` -> `real_export`.
3. `device_type` mapped to the controlled list; `null` (all `real_api` rows) or unknown -> `other` + `dq_flags`.
4. Metadata-driven DQ rules from `00_config` (quarantine vs flag); `user_id` must exist in `silver.users`; unknown `track_id` is only flagged (`is_catalog_matched = false`).
5. Row-level enrichment: `completion_rate`, `hour_of_day`, `day_of_week`, `is_weekend`, `session_id`, `play_sequence_num`.

In [ ]:
# %run ./00_config
# Placeholder Execution Flow:
# 1. Read the batches of bronze.events loaded in this run; keep the newest _batch_id per _partition_dt.
# 2. Apply cleaning rules and device mapping; split into valid_df and quarantine_df.
# 3. Enrich valid_df (completion_rate, time fields, sessions over affected days +-1).
# 4. MERGE valid_df into silver.events on event_id (update on _row_hash change; delete rows gone from backfilled days).
# 5. Append quarantine_df to silver.events_quarantine.
print("Silver events notebook initialized.")
